In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import numpy as np
import os

# ---------------------------------------------------------
# 1. Hyperparameters & Device configuration
# ---------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 50
IMAGE_SIZE = 128
DATA_DIR = './simpsons_dataset'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# ---------------------------------------------------------
# 2. Data Loading & Aggressive Augmentation
# ---------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
NUM_CLASSES = len(full_dataset.classes)
print(f"Total classes: {NUM_CLASSES}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------------------------------------------------------
# 3. Smoothed Class Weights
# ---------------------------------------------------------
train_indices = train_dataset.indices
train_labels = [full_dataset.targets[i] for i in train_indices]

class_sample_counts = np.bincount(train_labels)
smoothed_counts = np.sqrt(class_sample_counts)

total_samples = np.sum(smoothed_counts)
class_weights = total_samples / (NUM_CLASSES * (smoothed_counts + 1e-5))
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Smoothed Class Weights calculated successfully!")

# ---------------------------------------------------------
# 4. Transfer Learning: ResNet18 (WITH FREEZING)
# ---------------------------------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 🛑 THE FIX: Freeze all pre-trained layers
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features

# Replace the final layer. (This new layer automatically has requires_grad=True)
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

model = model.to(device)

# ---------------------------------------------------------
# 5. Loss, Optimizer, & Scheduler
# ---------------------------------------------------------
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# 🛑 THE FIX: Pass ONLY the unfrozen final layer parameters to the optimizer
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# ---------------------------------------------------------
# 6. Training Loop with Checkpointing
# ---------------------------------------------------------
def train_model():
    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        train_accuracy = 100 * correct_train / total_train
        
        # Validation Phase
        model.eval()
        correct_val = 0
        total_val = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_accuracy = 100 * correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_accuracy:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}% | "
              f"LR: {current_lr}")
              
        scheduler.step(avg_val_loss)

        if val_accuracy > best_val_acc:
            print(f"🌟 New High Score! Saving model... ({best_val_acc:.2f}% -> {val_accuracy:.2f}%)")
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), 'simpsons_resnet18_frozen.pth')

if __name__ == '__main__':
    train_model()
    print("Training complete! The best model is saved as 'simpsons_resnet18_frozen.pth'.")

Using device: cuda
Total classes: 43
Smoothed Class Weights calculated successfully!
Epoch [1/50] Train Loss: 2.6946, Train Acc: 37.66% | Val Loss: 2.3716, Val Acc: 38.15% | LR: 0.001
🌟 New High Score! Saving model... (0.00% -> 38.15%)
Epoch [2/50] Train Loss: 2.1530, Train Acc: 37.44% | Val Loss: 2.2127, Val Acc: 39.72% | LR: 0.001
🌟 New High Score! Saving model... (38.15% -> 39.72%)
Epoch [3/50] Train Loss: 2.0212, Train Acc: 36.87% | Val Loss: 2.1860, Val Acc: 34.12% | LR: 0.001
Epoch [4/50] Train Loss: 1.9634, Train Acc: 36.95% | Val Loss: 2.1357, Val Acc: 36.46% | LR: 0.001
Epoch [5/50] Train Loss: 1.9122, Train Acc: 36.84% | Val Loss: 2.0916, Val Acc: 36.04% | LR: 0.001
Epoch [6/50] Train Loss: 1.8866, Train Acc: 36.41% | Val Loss: 2.1318, Val Acc: 35.84% | LR: 0.001
Epoch [7/50] Train Loss: 1.8556, Train Acc: 37.06% | Val Loss: 2.1444, Val Acc: 31.67% | LR: 0.001
Epoch [8/50] Train Loss: 1.8280, Train Acc: 36.83% | Val Loss: 2.1547, Val Acc: 39.62% | LR: 0.001
Epoch [9/50] Train

KeyboardInterrupt: 